**HUMAN GENOME ::: DATASET**

![https://i.pinimg.com/originals/a3/35/1e/a3351e85adaacfb35197e6199f7d20e8.gif](https://i.pinimg.com/originals/a3/35/1e/a3351e85adaacfb35197e6199f7d20e8.gif)

In [2]:
!wget -c --progress=bar:force:noscroll https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz -O /kaggle/working/hg38.fa.gz

--2026-10-24 02:03:41--  https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 983659424 (938M) [application/x-gzip]
Saving to: ‘/kaggle/working/hg38.fa.gz’

/kaggle/working/hg3 100%[===================>] 938.09M  48.9MB/s    in 16s     

2026-10-24 02:03:58 (58.2 MB/s) - ‘/kaggle/working/hg38.fa.gz’ saved [983659424/983659424]



In [3]:
import gzip

with gzip.open("/kaggle/working/hg38.fa.gz", "rt") as f:
    for line in f:
        if line.startswith(">"):
            continue

        dna = line.strip().upper()
        if any(base in "ACGT" for base in dna):
            print(dna)
            break

TAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTA


**TRAINING SAMPLE ::: CHROMOSOME**

In [4]:
import gzip
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

PATH = "/kaggle/working/hg38.fa.gz"
CONTEXT = 256
BATCH_SIZE = 32

# 1. Load chromosome 1
parts = []
reading = False

with gzip.open(PATH, "rt") as f:
    for line in f:
        if line.startswith(">"):
            if reading:
                break
            reading = line[1:].split()[0] == "chr1"
        elif reading:
            parts.append(line.strip().upper())

if not parts:
    raise ValueError("chr1 was not found")

sequence = "".join(parts)
del parts

# 2. Encode bases: A=0, C=1, G=2, T=3; anything else=4
lookup = np.full(256, 4, dtype=np.uint8)
lookup[list(b"ACGT")] = np.arange(4, dtype=np.uint8)

tokens = lookup[np.frombuffer(sequence.encode("ascii"), dtype=np.uint8)]
del sequence

# 3. Split regions BEFORE creating windows
# Start small: 10 million bases for training, then 1 million for validation.
train_tokens = tokens[:10_000_000]
val_tokens = tokens[10_000_000:11_000_000]


class DNAWindows(Dataset):
    def __init__(self, tokens, context):
        # Each example needs context + 1 bases.
        width = context + 1
        count = len(tokens) // width
        windows = tokens[:count * width].reshape(count, width)

        # Keep only windows consisting entirely of A, C, G, T.
        self.windows = windows[(windows < 4).all(axis=1)].copy()

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        window = torch.tensor(self.windows[index], dtype=torch.long)
        return window[:-1], window[1:]


train_dataset = DNAWindows(train_tokens, CONTEXT)
val_dataset = DNAWindows(val_tokens, CONTEXT)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Training examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")

x, y = next(iter(train_loader))
alphabet = np.array(list("ACGT"))

print("Input shape:", x.shape)
print("Input: ", "".join(alphabet[x[0, :40].numpy()]))
print("Target:", "".join(alphabet[y[0, :40].numpy()]))

Training examples:   38,115
Validation examples: 3,891
Input shape: torch.Size([32, 256])
Input:  CACCTCCTGTGCTAATCCTTTAATTCTCAGCACTTAACAG
Target: ACCTCCTGTGCTAATCCTTTAATTCTCAGCACTTAACAGA


**HELIX-NEXT ::: TRANSFORMER**

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class HelixNext(nn.Module):
    def __init__(
        self,
        context=256,
        dim=128,
        heads=4,
        layers=4,
        dropout=0.1,
    ):
        super().__init__()

        self.context = context
        self.token_embedding = nn.Embedding(4, dim)
        self.position_embedding = nn.Embedding(context, dim)

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=heads,
                dim_feedforward=4 * dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(layers)
        ])

        self.norm = nn.LayerNorm(dim)
        self.output = nn.Linear(dim, 4)

        # True means "cannot attend to this position."
        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.ones(context, context, dtype=torch.bool),
                diagonal=1,
            ),
        )

    def forward(self, tokens, targets=None):
        _, length = tokens.shape

        if length > self.context:
            raise ValueError(f"Maximum context is {self.context}")

        positions = torch.arange(length, device=tokens.device)

        hidden = (
            self.token_embedding(tokens)
            + self.position_embedding(positions)
        )

        mask = self.causal_mask[:length, :length]

        for block in self.blocks:
            hidden = block(hidden, src_mask=mask)

        logits = self.output(self.norm(hidden))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, 4),
                targets.reshape(-1),
            )

        return logits, loss


model = HelixNext(context=CONTEXT).to(device)

parameter_count = sum(p.numel() for p in model.parameters())
print("Device:", device)
print(f"Parameters: {parameter_count:,}")

# Verify one forward pass.
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)

model.eval()
with torch.no_grad():
    logits, loss = model(x, y)

print("Output shape:", logits.shape)
print(f"Initial loss: {loss.item():.4f}")

Device: cuda
Parameters: 827,140
Output shape: torch.Size([32, 256, 4])
Initial loss: 1.4810


**TRAIN ::: HELIX-NEXT**

In [6]:
import math
from tqdm.auto import tqdm

EPOCHS = 3
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01,
)


@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0
    correct = 0
    total_bases = 0

    for x, y in tqdm(val_loader, desc="Validation", leave=False):
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)

        total_loss += loss.item() * y.numel()
        correct += (logits.argmax(dim=-1) == y).sum().item()
        total_bases += y.numel()

    return total_loss / total_bases, correct / total_bases


# Measure performance before training.
initial_val_loss, initial_accuracy = evaluate()
print(
    f"Before training | validation loss: {initial_val_loss:.4f} "
    f"| accuracy: {initial_accuracy:.2%}"
)

best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_bases = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS}",
    )

    for x, y in progress:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)

        _, loss = model(x, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * y.numel()
        total_bases += y.numel()

        progress.set_postfix(
            loss=f"{total_loss / total_bases:.4f}"
        )

    train_loss = total_loss / total_bases
    val_loss, val_accuracy = evaluate()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })

    print(
        f"Epoch {epoch} | train loss: {train_loss:.4f} "
        f"| val loss: {val_loss:.4f} "
        f"| val accuracy: {val_accuracy:.2%} "
        f"| val perplexity: {math.exp(val_loss):.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": {
                "context": CONTEXT,
                "dim": 128,
                "heads": 4,
                "layers": 4,
                "dropout": 0.1,
            },
            "token_to_id": {"A": 0, "C": 1, "G": 2, "T": 3},
            "epoch": epoch,
            "val_loss": val_loss,
            "history": history.copy(),
        }, "/kaggle/working/helixnext_best.pt")

        print("Saved best checkpoint.")

Validation:   0%|          | 0/122 [00:00<?, ?it/s]

Before training | validation loss: 1.4927 | accuracy: 25.89%


Epoch 1/3:   0%|          | 0/1192 [00:00<?, ?it/s]

Validation:   0%|          | 0/122 [00:00<?, ?it/s]

Epoch 1 | train loss: 1.3220 | val loss: 1.3193 | val accuracy: 35.33% | val perplexity: 3.741
Saved best checkpoint.


...

Validation:   0%|          | 0/122 [00:00<?, ?it/s]

Epoch 2 | train loss: 1.3133 | val loss: 1.3145 | val accuracy: 35.85% | val perplexity: 3.723
Saved best checkpoint.


Epoch 3/3:   0%|          | 0/1192 [00:00<?, ?it/s]

Validation:   0%|          | 0/122 [00:00<?, ?it/s]

Epoch 3 | train loss: 1.3100 | val loss: 1.3115 | val accuracy: 36.13% | val perplexity: 3.712
Saved best checkpoint.


*Training and validation losses are close, so these results show no obvious overfitting yet. But we still need to check whether the transformer beats simple DNA statistics.*

**STATISTICAL PREDICTION ::: DNA SEQUENCE**

In [7]:
import numpy as np

# The exact input/target pairs used by your datasets.
train_x = train_dataset.windows[:, :-1]
train_y = train_dataset.windows[:, 1:]

val_x = val_dataset.windows[:, :-1]
val_y = val_dataset.windows[:, 1:]

# 1. Base-frequency baseline, with add-one smoothing.
base_counts = np.bincount(
    train_y.reshape(-1), minlength=4
).astype(np.float64) + 1

base_probs = base_counts / base_counts.sum()

base_loss = -np.log(base_probs[val_y]).mean()
base_accuracy = (val_y == base_probs.argmax()).mean()

# 2. Bigram baseline: P(next base | current base).
pair_ids = (
    train_x.reshape(-1).astype(np.int64) * 4
    + train_y.reshape(-1)
)

pair_counts = np.bincount(
    pair_ids, minlength=16
).reshape(4, 4).astype(np.float64) + 1

pair_probs = pair_counts / pair_counts.sum(axis=1, keepdims=True)

bigram_loss = -np.log(pair_probs[val_x, val_y]).mean()
bigram_accuracy = (
    pair_probs.argmax(axis=1)[val_x] == val_y
).mean()

print(f"{'Model':<18} {'Val loss':>10} {'Accuracy':>10}")
print(f"{'Base frequency':<18} {base_loss:>10.4f} {base_accuracy:>10.2%}")
print(f"{'Bigram':<18} {bigram_loss:>10.4f} {bigram_accuracy:>10.2%}")

best = min(history, key=lambda row: row["val_loss"])
print(
    f"{'HelixNext':<18} "
    f"{best['val_loss']:>10.4f} "
    f"{best['val_accuracy']:>10.2%}"
)

Model                Val loss   Accuracy
Base frequency         1.3872     23.92%
Bigram                 1.3526     29.46%
HelixNext              1.3115     36.13%


*HelixNext beats both baselines on your validation region. Helix-Next transformer predicts unseen DNA better than base frequencies or a one-base-context model.*

**HELIX-NEXT ::: WITH DIFFERENT CONTEXT LENGTHS**

In [8]:
import torch
import pandas as pd
from tqdm.auto import tqdm

# Restore the best saved model.
checkpoint = torch.load(
    "/kaggle/working/helixnext_best.pt",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state"])
model.eval()

CONTEXT_LENGTHS = [1, 8, 32, 128, 256]


@torch.no_grad()
def predict_with_context(x, context_length):
    full_length = x.shape[1]

    # Keep only the final context_length input bases.
    tokens = x[:, -context_length:]

    # Preserve their original position indices.
    positions = torch.arange(
        full_length - context_length,
        full_length,
        device=x.device,
    )

    hidden = (
        model.token_embedding(tokens)
        + model.position_embedding(positions)
    )

    mask = model.causal_mask[:context_length, :context_length]

    for block in model.blocks:
        hidden = block(hidden, src_mask=mask)

    # Predict only the base AFTER the final input position.
    return model.output(model.norm(hidden[:, -1]))


results = []

for context_length in tqdm(CONTEXT_LENGTHS, desc="Testing context lengths"):
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for x, y in val_loader:
        x = x.to(device)
        target = y[:, -1].to(device)

        logits = predict_with_context(x, context_length)

        total_loss += torch.nn.functional.cross_entropy(
            logits, target, reduction="sum"
        ).item()

        total_correct += (
            logits.argmax(dim=-1) == target
        ).sum().item()

        total_examples += target.numel()

    results.append({
        "Context bases": context_length,
        "Loss": total_loss / total_examples,
        "Accuracy (%)": 100 * total_correct / total_examples,
        "Examples": total_examples,
    })

df = pd.DataFrame(results)
print(df.to_string(
    index=False,
    formatters={
        "Loss": "{:.4f}".format,
        "Accuracy (%)": "{:.2f}".format,
    },
))

Testing context lengths:   0%|          | 0/5 [00:00<?, ?it/s]

 Context bases   Loss Accuracy (%)  Examples
             1 1.6360        29.56      3891
             8 1.4342        32.20      3891
            32 1.3347        34.10      3891
           128 1.3314        33.46      3891
           256 1.3060        36.19      3891


**HELIX-NEXT ::: TESTING ON UNSEEN REGION**

In [9]:
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Existing `tokens` contains the full chromosome 1.
TEST_START = 20_000_000
TEST_END = 21_000_000

assert len(tokens) >= TEST_END, "Full chromosome 1 tokens are required."

test_dataset = DNAWindows(
    tokens[TEST_START:TEST_END],
    CONTEXT,
)

if len(test_dataset) == 0:
    raise ValueError("No valid DNA windows in the test region.")

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Restore the checkpoint selected using validation data.
checkpoint = torch.load(
    "/kaggle/working/helixnext_best.pt",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state"])
model.eval()

# Fit baselines using TRAINING data only.
train_x = train_dataset.windows[:, :-1].reshape(-1)
train_y = train_dataset.windows[:, 1:].reshape(-1)

base_counts = np.bincount(
    train_y, minlength=4
).astype(np.float64) + 1

base_probs = base_counts / base_counts.sum()

pair_ids = train_x.astype(np.int64) * 4 + train_y
pair_counts = np.bincount(
    pair_ids, minlength=16
).reshape(4, 4).astype(np.float64) + 1

pair_probs = pair_counts / pair_counts.sum(axis=1, keepdims=True)

# Evaluate baselines on every target position.
test_x = test_dataset.windows[:, :-1]
test_y = test_dataset.windows[:, 1:]

base_loss = float(-np.log(base_probs[test_y]).mean())
base_accuracy = float((test_y == base_probs.argmax()).mean())

bigram_loss = float(-np.log(pair_probs[test_x, test_y]).mean())
bigram_accuracy = float(
    (pair_probs.argmax(axis=1)[test_x] == test_y).mean()
)

# Evaluate HelixNext on the exact same target positions.
total_loss = 0.0
total_correct = 0
total_targets = 0

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Untouched-region test"):
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)

        total_loss += F.cross_entropy(
            logits.reshape(-1, 4),
            y.reshape(-1),
            reduction="sum",
        ).item()

        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_targets += y.numel()

model_loss = total_loss / total_targets
model_accuracy = total_correct / total_targets

results = pd.DataFrame([
    {
        "Model": "Base frequency",
        "Test loss": base_loss,
        "Accuracy (%)": 100 * base_accuracy,
    },
    {
        "Model": "Bigram",
        "Test loss": bigram_loss,
        "Accuracy (%)": 100 * bigram_accuracy,
    },
    {
        "Model": "HelixNext",
        "Test loss": model_loss,
        "Accuracy (%)": 100 * model_accuracy,
    },
])

print(f"\nTest region: chr1[{TEST_START:,}:{TEST_END:,}] (0-based)")
print(f"Valid windows: {len(test_dataset):,}")
print(f"Target bases evaluated: {total_targets:,}\n")

print(results.to_string(
    index=False,
    formatters={
        "Test loss": "{:.4f}".format,
        "Accuracy (%)": "{:.2f}".format,
    },
))

print(
    "\nHelixNext accuracy gain over bigram: "
    f"{100 * (model_accuracy - bigram_accuracy):+.2f} percentage points"
)

Untouched-region test:   0%|          | 0/122 [00:00<?, ?it/s]


Test region: chr1[20,000,000:21,000,000] (0-based)
Valid windows: 3,891
Target bases evaluated: 996,096

         Model Test loss Accuracy (%)
Base frequency    1.3875        23.02
        Bigram    1.3515        28.32
     HelixNext    1.3151        35.39

HelixNext accuracy gain over bigram: +7.08 percentage points


**BENCHMARK - I**

Trained an 827K-parameter causal DNA transformer from scratch. On a held-out chromosome 1 region, it achieved 35.39% next-base accuracy, outperforming a bigram baseline by 7.08 percentage points.

**HELIX-NEXT ::: TRAINING FOR MORE EPOCHS**

In [11]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

CHECKPOINT = "/kaggle/working/helixnext_best.pt"
MAX_EXTRA_EPOCHS = 10
EARLY_STOP_PATIENCE = 3
MIN_IMPROVEMENT = 1e-4

# Resume model AND optimizer.
checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01,
)
optimizer.load_state_dict(checkpoint["optimizer_state"])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=0,
    threshold=MIN_IMPROVEMENT,
    threshold_mode="abs",
    min_lr=1e-6,
)

best_val_loss = checkpoint["val_loss"]
start_epoch = checkpoint["epoch"] + 1
history = checkpoint.get("history", []).copy()

# Start scheduling relative to the checkpoint's validation loss.
scheduler.step(best_val_loss)

# Separate early-stopping threshold from best-checkpoint selection.
early_stop_best = best_val_loss
epochs_without_improvement = 0


@torch.no_grad()
def validate():
    model.eval()
    loss_sum = 0.0
    correct = 0
    count = 0

    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)

        loss_sum += F.cross_entropy(
            logits.reshape(-1, 4),
            y.reshape(-1),
            reduction="sum",
        ).item()

        correct += (logits.argmax(dim=-1) == y).sum().item()
        count += y.numel()

    return loss_sum / count, correct / count


print(f"Resuming from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {best_val_loss:.4f}")

for epoch in range(start_epoch, start_epoch + MAX_EXTRA_EPOCHS):
    model.train()
    loss_sum = 0.0
    count = 0
    epoch_lr = optimizer.param_groups[0]["lr"]

    progress = tqdm(train_loader, desc=f"Epoch {epoch}")

    for x, y in progress:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        loss_sum += loss.item() * y.numel()
        count += y.numel()

        progress.set_postfix(
            loss=f"{loss_sum / count:.4f}",
            lr=f"{epoch_lr:.1e}",
        )

    train_loss = loss_sum / count
    val_loss, val_accuracy = validate()
    scheduler.step(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "lr": epoch_lr,
    })

    print(
        f"Epoch {epoch} | train: {train_loss:.4f} "
        f"| val: {val_loss:.4f} "
        f"| val accuracy: {val_accuracy:.2%}"
    )

    # Save every new lowest validation loss.
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "config": checkpoint["config"],
            "token_to_id": checkpoint["token_to_id"],
            "epoch": epoch,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "history": history.copy(),
        }, CHECKPOINT)

        print("Saved best checkpoint.")

    if val_loss < early_stop_best - MIN_IMPROVEMENT:
        early_stop_best = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    next_lr = optimizer.param_groups[0]["lr"]
    if next_lr < epoch_lr:
        print(f"Learning rate reduced to {next_lr:.1e}")

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print("Early stopping: validation loss has stalled.")
        break

# Restore the best model, rather than leaving the last epoch loaded.
best_checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=True,
)
model.load_state_dict(best_checkpoint["model_state"])
model.eval()

print(
    f"\nRestored best epoch: {best_checkpoint['epoch']} "
    f"| validation loss: {best_checkpoint['val_loss']:.4f}"
)

Resuming from epoch 13
Best validation loss: 1.1865


Epoch 14:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 14 | train: 1.2426 | val: 1.1787 | val accuracy: 45.09%
Saved best checkpoint.


Epoch 15:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 15 | train: 1.2365 | val: 1.1672 | val accuracy: 45.75%
Saved best checkpoint.


...

Epoch 21 | train: 1.2131 | val: 1.1377 | val accuracy: 47.13%
Saved best checkpoint.


Epoch 22:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 22 | train: 1.2101 | val: 1.1353 | val accuracy: 47.32%
Saved best checkpoint.


Epoch 23:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 23 | train: 1.2068 | val: 1.1304 | val accuracy: 47.53%
Saved best checkpoint.

Restored best epoch: 23 | validation loss: 1.1304


*Let’s try a causal convolution + transformer hybrid. The Latest result is 47.53% validation accuracy, up from 44.85%, but the gains are slowing.*

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class CausalConvBlock(nn.Module):
    def __init__(self, dim, kernel_size=7, dropout=0.1):
        super().__init__()
        self.left_padding = kernel_size - 1
        self.norm = nn.LayerNorm(dim)

        # Local pattern processing, followed by channel mixing.
        self.conv = nn.Conv1d(
            dim, dim, kernel_size=kernel_size, groups=dim
        )
        self.mix = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        hidden = self.norm(x).transpose(1, 2)

        # Only pad on the LEFT: no future-base leakage.
        hidden = F.pad(hidden, (self.left_padding, 0))
        hidden = self.conv(hidden).transpose(1, 2)
        hidden = self.mix(F.gelu(hidden))

        return x + self.dropout(hidden)


class HelixNextV2(nn.Module):
    def __init__(
        self,
        context=256,
        dim=128,
        heads=4,
        layers=4,
        kernel_size=7,
        dropout=0.1,
    ):
        super().__init__()
        self.context = context
        self.token_embedding = nn.Embedding(4, dim)
        self.position_embedding = nn.Embedding(context, dim)

        self.convs = nn.ModuleList([
            CausalConvBlock(dim, kernel_size, dropout)
            for _ in range(layers)
        ])

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=heads,
                dim_feedforward=4 * dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(layers)
        ])

        self.norm = nn.LayerNorm(dim)
        self.output = nn.Linear(dim, 4)

        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.ones(context, context, dtype=torch.bool),
                diagonal=1,
            ),
        )

    def forward(self, tokens, targets=None):
        length = tokens.shape[1]
        if not 1 <= length <= self.context:
            raise ValueError(f"Input length must be 1–{self.context}")

        positions = torch.arange(length, device=tokens.device)
        hidden = (
            self.token_embedding(tokens)
            + self.position_embedding(positions)
        )
        mask = self.causal_mask[:length, :length]

        for conv, block in zip(self.convs, self.blocks):
            hidden = conv(hidden)
            hidden = block(hidden, src_mask=mask)

        logits = self.output(self.norm(hidden))
        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, 4),
                targets.reshape(-1),
            )

        return logits, loss


v2_config = {
    "context": CONTEXT,
    "dim": 128,
    "heads": 4,
    "layers": 4,
    "kernel_size": 7,
    "dropout": 0.1,
}

model_v2 = HelixNextV2(**v2_config).to(device)

# Verify that changing future inputs cannot change earlier predictions.
model_v2.eval()
with torch.no_grad():
    original = torch.randint(0, 4, (2, CONTEXT), device=device)
    changed = original.clone()
    cut = CONTEXT // 2
    changed[:, cut:] = (changed[:, cut:] + 1) % 4

    original_logits, _ = model_v2(original)
    changed_logits, _ = model_v2(changed)

    torch.testing.assert_close(
        original_logits[:, :cut],
        changed_logits[:, :cut],
        atol=1e-5,
        rtol=1e-5,
    )

print("Causality check passed.")
print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in model_v2.parameters()):,}")

Causality check passed.
Device: cuda
Parameters: 898,308


In [13]:
from pathlib import Path
from tqdm.auto import tqdm

V2_CHECKPOINT = "/kaggle/working/helixnext_v2_best.pt"
EPOCHS = 23

# Prevent accidentally replacing a previous v2 experiment.
if Path(V2_CHECKPOINT).exists():
    raise FileExistsError(
        "A v2 checkpoint already exists. Choose a new filename "
        "for a fresh experiment."
    )

optimizer_v2 = torch.optim.AdamW(
    model_v2.parameters(),
    lr=3e-4,
    weight_decay=0.01,
)

best_v2_loss = float("inf")
v2_history = []


@torch.no_grad()
def evaluate_v2():
    model_v2.eval()
    loss_sum = 0.0
    correct = 0
    total = 0

    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        logits, loss = model_v2(x, y)

        loss_sum += loss.item() * y.numel()
        correct += (logits.argmax(-1) == y).sum().item()
        total += y.numel()

    return loss_sum / total, correct / total


for epoch in range(1, EPOCHS + 1):
    model_v2.train()
    loss_sum = 0.0
    total = 0

    progress = tqdm(train_loader, desc=f"V2 epoch {epoch}/{EPOCHS}")

    for x, y in progress:
        x, y = x.to(device), y.to(device)

        optimizer_v2.zero_grad(set_to_none=True)
        _, loss = model_v2(x, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
        optimizer_v2.step()

        loss_sum += loss.item() * y.numel()
        total += y.numel()
        progress.set_postfix(loss=f"{loss_sum / total:.4f}")

    train_loss = loss_sum / total
    val_loss, val_accuracy = evaluate_v2()

    v2_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })

    print(
        f"Epoch {epoch} | train: {train_loss:.4f} "
        f"| val: {val_loss:.4f} "
        f"| val accuracy: {val_accuracy:.2%}"
    )

    if val_loss < best_v2_loss:
        best_v2_loss = val_loss

        torch.save({
            "architecture": "HelixNextV2",
            "model_state": model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "config": v2_config,
            "token_to_id": {"A": 0, "C": 1, "G": 2, "T": 3},
            "epoch": epoch,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "history": v2_history.copy(),
        }, V2_CHECKPOINT)

        print("Saved best v2 checkpoint.")

best_v2 = torch.load(
    V2_CHECKPOINT,
    map_location=device,
    weights_only=True,
)
model_v2.load_state_dict(best_v2["model_state"])
model_v2.eval()

print(
    f"\nBest V2: epoch {best_v2['epoch']} "
    f"| loss: {best_v2['val_loss']:.4f} "
    f"| accuracy: {best_v2['val_accuracy']:.2%}"
)
print("V1 reference: validation loss 1.1304 | accuracy 47.53%")

V2 epoch 1/23:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 1 | train: 1.3098 | val: 1.2990 | val accuracy: 37.88%
Saved best v2 checkpoint.


V2 epoch 2/23:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 2 | train: 1.2885 | val: 1.2296 | val accuracy: 43.05%
Saved best v2 checkpoint.


V2 epoch 3/23:   0%|          | 0/1192 [00:00<?, ?it/s]

...

Epoch 10 | train: 1.1643 | val: 1.0944 | val accuracy: 49.29%
Saved best v2 checkpoint.


V2 epoch 11/23:   0%|          | 0/1192 [00:00<?, ?it/s]

Epoch 11 | train: 1.1623 | val: 1.0939 | val accuracy: 49.29%
Saved best v2 checkpoint.


V2 epoch 12/23:   0%|          | 0/1192 [00:00<?, ?it/s]

KeyboardInterrupt: 

*I'm early-stopping............*